In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

## Helper Classes

In [ ]:
class BayesianABTest:
    def __init__(self, str_uri, str_dirname_output, int_n_simulations=100000):
        self.str_uri = str_uri
        self.str_dirname_output = str_dirname_output
        self.int_n_simulations = int_n_simulations
        self.df_data = None
        self.dict_results = {}

    def import_data(self):
        self.df_data = pd.read_parquet(self.str_uri)
        print(f'Data loaded: {self.df_data.shape[0]:,} rows')
        return self

    def run_bayesian_test(self, str_metric, flt_prior_alpha=1, flt_prior_beta=1):
        df_control = self.df_data[self.df_data['version'] == 'gate_30']
        df_treatment = self.df_data[self.df_data['version'] == 'gate_40']
        int_successes_control = df_control[str_metric].sum()
        int_failures_control = len(df_control) - int_successes_control
        int_successes_treatment = df_treatment[str_metric].sum()
        int_failures_treatment = len(df_treatment) - int_successes_treatment
        flt_alpha_control = flt_prior_alpha + int_successes_control
        flt_beta_control = flt_prior_beta + int_failures_control
        flt_alpha_treatment = flt_prior_alpha + int_successes_treatment
        flt_beta_treatment = flt_prior_beta + int_failures_treatment
        np.random.seed(42)
        arr_samples_control = np.random.beta(flt_alpha_control, flt_beta_control, self.int_n_simulations)
        arr_samples_treatment = np.random.beta(flt_alpha_treatment, flt_beta_treatment, self.int_n_simulations)
        arr_diff = arr_samples_treatment - arr_samples_control
        flt_prob_treatment_better = (arr_diff > 0).mean()
        flt_prob_control_better = (arr_diff < 0).mean()
        flt_expected_loss_treatment = arr_diff[arr_diff < 0].mean() if (arr_diff < 0).any() else 0
        flt_expected_loss_control = arr_diff[arr_diff > 0].mean() if (arr_diff > 0).any() else 0
        flt_hdi_lower = np.percentile(arr_diff, 2.5)
        flt_hdi_upper = np.percentile(arr_diff, 97.5)
        str_day = '1-Day' if str_metric == 'retention_1' else '7-Day'
        print(f'\n=== BAYESIAN TEST: {str_day} Retention ===')
        print(f'Prior: Beta({flt_prior_alpha}, {flt_prior_beta}) (uninformative)')
        print(f'\nControl posterior: Beta({flt_alpha_control}, {flt_beta_control})')
        print(f'  Mean: {flt_alpha_control / (flt_alpha_control + flt_beta_control):.4f}')
        print(f'Treatment posterior: Beta({flt_alpha_treatment}, {flt_beta_treatment})')
        print(f'  Mean: {flt_alpha_treatment / (flt_alpha_treatment + flt_beta_treatment):.4f}')
        print(f'\nP(Control > Treatment): {flt_prob_control_better:.4f} ({flt_prob_control_better*100:.2f}%)')
        print(f'P(Treatment > Control): {flt_prob_treatment_better:.4f} ({flt_prob_treatment_better*100:.2f}%)')
        print(f'\n95% HDI for difference: [{flt_hdi_lower:.4f}, {flt_hdi_upper:.4f}]')
        print(f'Expected loss if choosing Treatment: {abs(flt_expected_loss_treatment)*100:.4f} pp')
        print(f'Expected loss if choosing Control: {abs(flt_expected_loss_control)*100:.4f} pp')
        self.dict_results[str_metric] = {
            'str_metric': str_day,
            'arr_samples_control': arr_samples_control,
            'arr_samples_treatment': arr_samples_treatment,
            'arr_diff': arr_diff,
            'flt_prob_control_better': flt_prob_control_better,
            'flt_prob_treatment_better': flt_prob_treatment_better,
            'flt_hdi_lower': flt_hdi_lower,
            'flt_hdi_upper': flt_hdi_upper,
            'flt_expected_loss_treatment': flt_expected_loss_treatment,
            'flt_expected_loss_control': flt_expected_loss_control,
            'flt_alpha_control': flt_alpha_control,
            'flt_beta_control': flt_beta_control,
            'flt_alpha_treatment': flt_alpha_treatment,
            'flt_beta_treatment': flt_beta_treatment
        }
        return self

    def plot_posteriors(self, str_metric):
        dict_r = self.dict_results[str_metric]
        fig, ax = plt.subplots(figsize=(12, 6))
        arr_x = np.linspace(
            min(dict_r['arr_samples_control'].min(), dict_r['arr_samples_treatment'].min()),
            max(dict_r['arr_samples_control'].max(), dict_r['arr_samples_treatment'].max()),
            1000
        )
        arr_pdf_control = stats.beta.pdf(arr_x, dict_r['flt_alpha_control'], dict_r['flt_beta_control'])
        arr_pdf_treatment = stats.beta.pdf(arr_x, dict_r['flt_alpha_treatment'], dict_r['flt_beta_treatment'])
        ax.plot(arr_x, arr_pdf_control, color='#4C72B0', linewidth=2, label='Control (gate_30)')
        ax.fill_between(arr_x, arr_pdf_control, alpha=0.3, color='#4C72B0')
        ax.plot(arr_x, arr_pdf_treatment, color='#DD8452', linewidth=2, label='Treatment (gate_40)')
        ax.fill_between(arr_x, arr_pdf_treatment, alpha=0.3, color='#DD8452')
        ax.set_title(f'Posterior Distributions — {dict_r["str_metric"]} Retention', fontsize=14, y=1.02)
        ax.set_xlabel('Retention Rate')
        ax.set_ylabel('Probability Density')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        str_suffix = '1' if str_metric == 'retention_1' else '7'
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/0{str_suffix}_posteriors_{str_metric}.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def plot_difference_distribution(self, str_metric):
        dict_r = self.dict_results[str_metric]
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.hist(dict_r['arr_diff'] * 100, bins=100, color='#4C72B0', edgecolor='black', alpha=0.7, density=True)
        ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='No Effect')
        ax.axvline(x=dict_r['flt_hdi_lower'] * 100, color='green', linestyle='--',
                   label=f'95% HDI [{dict_r["flt_hdi_lower"]*100:.2f}, {dict_r["flt_hdi_upper"]*100:.2f}]')
        ax.axvline(x=dict_r['flt_hdi_upper'] * 100, color='green', linestyle='--')
        ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 1],
                          dict_r['flt_hdi_lower'] * 100, dict_r['flt_hdi_upper'] * 100,
                          alpha=0.15, color='green')
        ax.set_title(f'Distribution of Treatment Effect — {dict_r["str_metric"]} Retention', fontsize=14, y=1.02)
        ax.set_xlabel('Difference in Retention Rate (Percentage Points)')
        ax.set_ylabel('Density')
        flt_prob = dict_r['flt_prob_control_better']
        ax.text(0.02, 0.95, f'P(Control > Treatment) = {flt_prob:.2%}',
                transform=ax.transAxes, fontsize=11, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        str_suffix = '2' if str_metric == 'retention_1' else '4'
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/0{str_suffix}_difference_{str_metric}.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def plot_cumulative_probability(self, str_metric):
        dict_r = self.dict_results[str_metric]
        df_control = self.df_data[self.df_data['version'] == 'gate_30']
        df_treatment = self.df_data[self.df_data['version'] == 'gate_40']
        list_n = np.arange(100, len(df_control) + 1, 500)
        list_prob = []
        for int_n in list_n:
            int_s_c = df_control[str_metric].iloc[:int_n].sum()
            int_f_c = int_n - int_s_c
            int_s_t = df_treatment[str_metric].iloc[:int_n].sum()
            int_f_t = int_n - int_s_t
            arr_c = np.random.beta(1 + int_s_c, 1 + int_f_c, 10000)
            arr_t = np.random.beta(1 + int_s_t, 1 + int_f_t, 10000)
            list_prob.append((arr_c > arr_t).mean())
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.plot(list_n, list_prob, color='#4C72B0', linewidth=2)
        ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
        ax.axhline(y=0.95, color='red', linestyle='--', alpha=0.7, label='95% Threshold')
        ax.set_title(f'P(Control > Treatment) Over Time — {dict_r["str_metric"]} Retention', fontsize=14, y=1.02)
        ax.set_xlabel('Players Observed per Group')
        ax.set_ylabel('P(Control > Treatment)')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1)
        str_suffix = '5' if str_metric == 'retention_1' else '6'
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/0{str_suffix}_cumulative_{str_metric}.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def save_results(self):
        list_rows = []
        for str_key, dict_r in self.dict_results.items():
            list_rows.append({
                'metric': dict_r['str_metric'],
                'p_control_better': dict_r['flt_prob_control_better'],
                'p_treatment_better': dict_r['flt_prob_treatment_better'],
                'hdi_lower': dict_r['flt_hdi_lower'],
                'hdi_upper': dict_r['flt_hdi_upper'],
                'expected_loss_treatment': dict_r['flt_expected_loss_treatment'],
                'expected_loss_control': dict_r['flt_expected_loss_control']
            })
        df_results = pd.DataFrame(list_rows)
        df_results.to_csv(f'{self.str_dirname_output}/bayesian_results.csv', index=False)
        print(f'\nResults saved to {self.str_dirname_output}/bayesian_results.csv')
        print(df_results.to_string(index=False))
        return self

## Constants

In [ ]:
str_bucket = 'ab-testing-demo-repo'
str_task = 'bayesian_test'
str_dirname_output = './output'
str_uri = f's3://{str_bucket}/00_data_collection/cookie_cats.parquet'
int_n_simulations = 100000

## Output Directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass
print(f'Output directory ready: {str_dirname_output}')

## Run Bayesian Tests

In [ ]:
cls_bayes = BayesianABTest(str_uri, str_dirname_output, int_n_simulations=int_n_simulations)
cls_bayes.import_data()

In [ ]:
cls_bayes.run_bayesian_test('retention_1')

In [ ]:
cls_bayes.plot_posteriors('retention_1')

In [ ]:
cls_bayes.plot_difference_distribution('retention_1')

In [ ]:
cls_bayes.run_bayesian_test('retention_7')

In [ ]:
cls_bayes.plot_posteriors('retention_7')

In [ ]:
cls_bayes.plot_difference_distribution('retention_7')

In [ ]:
cls_bayes.plot_cumulative_probability('retention_7')

In [ ]:
cls_bayes.save_results()

## Completion

In [ ]:
print('\n=== BAYESIAN TESTING COMPLETE ===')
print(f'Results and visualizations saved to: {str_dirname_output}')